# 02 – Discrete Distributions

Author: Joe  
Project: AI Foundations – Probability & Statistics

This notebook goes deeper into **discrete random variables** and their distributions.
We'll focus on:

- Bernoulli
- Binomial
- Geometric
- Poisson

The goals here are:
- connect formulas to intuition,
- verify results with simulation,
- and link concepts to real-world scenarios.

## Contents
1. [Setup](#1-setup)
2. [Bernoulli Distribution](#2-bernoulli-distribution)
3. [Binomial Distribution](#3-binomial-distribution)
4. [Geometric Distribution](#4-geometric-distribution)
5. [Poisson Distribution](#5-poisson-distribution)
6. [Comparing Discrete Distributions](#6-comparing-discrete-distributions)
7. [Practice / TODOs](#7-practice--todos)


## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

np.set_printoptions(precision=4, suppress=True)

---
## 2. Bernoulli Distribution

**Story:** A single trial with only two outcomes: success (1) and failure (0).

Let $X \sim \text{Bernoulli}(p)$, where $p = P(X=1)$.

**PMF (probability mass function)**:
\begin{equation}
P(X = x) = \begin{cases}
p & x = 1, \\
1 - p & x = 0, \\
0 & \text{otherwise.}
\end{cases}
\end{equation}

**Mean & variance**:
- $\mathbb{E}[X] = p$
- $\text{Var}(X) = p(1-p)$

In [ ]:
# Bernoulli pmf helper
def bernoulli_pmf(x, p):
    if x == 1:
        return p
    if x == 0:
        return 1 - p
    return 0.0

for p in [0.2, 0.5, 0.8]:
    print(f"p = {p}")
    print("P(X=0) =", bernoulli_pmf(0, p))
    print("P(X=1) =", bernoulli_pmf(1, p))
    print()

In [ ]:
# Simulate Bernoulli and compare empirical mean/variance to theory
def simulate_bernoulli(p=0.3, n=10000):
    samples = (np.random.rand(n) < p).astype(int)
    empirical_mean = samples.mean()
    empirical_var = samples.var(ddof=1)
    return samples, empirical_mean, empirical_var

p = 0.3
samples_bern, mean_bern, var_bern = simulate_bernoulli(p=p, n=50000)

print("Theoretical mean:", p)
print("Empirical mean:", mean_bern)
print("Theoretical variance:", p * (1 - p))
print("Empirical variance:", var_bern)

---
## 3. Binomial Distribution

**Story:** Number of successes in $n$ independent Bernoulli$(p)$ trials.

Let $X \sim \text{Binomial}(n, p)$.

**PMF:**
\begin{equation}
P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0,1,\dots,n.
\end{equation}

**Mean & variance**:
- $\mathbb{E}[X] = np$
- $\text{Var}(X) = np(1-p)$

In [ ]:
# Binomial pmf implementation
from math import comb

def binomial_pmf(k, n, p):
    if k < 0 or k > n:
        return 0.0
    return comb(n, k) * (p ** k) * ((1 - p) ** (n - k))

n = 10
p = 0.4
ks = np.arange(0, n+1)
probs = np.array([binomial_pmf(k, n, p) for k in ks])

print("Sum of pmf (should be 1):", probs.sum())

In [ ]:
# Plot Binomial(10, 0.4) pmf
plt.stem(ks, probs, use_line_collection=True)
plt.xlabel('k')
plt.ylabel('P(X = k)')
plt.title('Binomial(10, 0.4) pmf')
plt.show()

In [ ]:
# Simulate Binomial and compare mean/variance
n_trials = 10
p = 0.4
samples_binom = np.random.binomial(n_trials, p, size=50000)

mean_emp = samples_binom.mean()
var_emp = samples_binom.var(ddof=1)

print("Theoretical mean:", n_trials * p)
print("Empirical mean:", mean_emp)
print("Theoretical variance:", n_trials * p * (1 - p))
print("Empirical variance:", var_emp)

---
## 4. Geometric Distribution

**Story:** Number of trials **until the first success** in independent Bernoulli$(p)$ trials.

Here we use the convention: $X = 1,2,3,\dots$ counts the trial on which the first success occurs.

Let $X \sim \text{Geometric}(p)$.

**PMF:**
\begin{equation}
P(X = k) = (1-p)^{k-1} p, \quad k = 1,2,3,\dots
\end{equation}

**Mean & variance**:
- $\mathbb{E}[X] = \dfrac{1}{p}$
- $\text{Var}(X) = \dfrac{1-p}{p^2}$

In [ ]:
# Geometric pmf and simulation
def geometric_pmf(k, p):
    if k < 1:
        return 0.0
    return ((1 - p) ** (k - 1)) * p

p = 0.3
ks = np.arange(1, 15)
probs_geo = np.array([geometric_pmf(k, p) for k in ks])

print("Sum of pmf up to 14:", probs_geo.sum())
print("(The true sum over k=1..infinity is exactly 1)")

In [ ]:
# Plot Geometric(p=0.3) pmf (truncated at k=14)
plt.stem(ks, probs_geo, use_line_collection=True)
plt.xlabel('k (trial of first success)')
plt.ylabel('P(X = k)')
plt.title('Geometric(p=0.3) pmf (truncated)')
plt.show()

In [ ]:
# Simulate Geometric(p)
def simulate_geometric(p=0.3, n_samples=50000):
    # Inverse transform: X = ceil(log(1-U)/log(1-p))
    U = np.random.rand(n_samples)
    X = np.ceil(np.log(1 - U) / np.log(1 - p)).astype(int)
    return X

p = 0.3
samples_geo = simulate_geometric(p=p, n_samples=50000)

mean_emp = samples_geo.mean()
var_emp = samples_geo.var(ddof=1)

print("Theoretical mean:", 1 / p)
print("Empirical mean:", mean_emp)
print("Theoretical variance:", (1 - p) / (p ** 2))
print("Empirical variance:", var_emp)

---
## 5. Poisson Distribution

**Story:** Counts of events in a fixed interval when events occur randomly and independently
at an average rate $\lambda$.

Let $X \sim \text{Poisson}(\lambda)$.

**PMF:**
\begin{equation}
P(X = k) = e^{-\lambda} \frac{\lambda^k}{k!}, \quad k = 0,1,2,\dots
\end{equation}

**Mean & variance**:
- $\mathbb{E}[X] = \lambda$
- $\text{Var}(X) = \lambda$

Examples:
- Number of emails per hour,
- Number of defects on a length of cable,
- Number of arrivals in a queueing system.

In [ ]:
# Poisson pmf implementation
def poisson_pmf(k, lam):
    if k < 0:
        return 0.0
    return math.exp(-lam) * (lam ** k) / math.factorial(k)

lam = 3.0
ks = np.arange(0, 15)
probs_pois = np.array([poisson_pmf(k, lam) for k in ks])

print("Sum of pmf up to 14:", probs_pois.sum())
print("(The true sum over k=0..infinity is exactly 1)")

In [ ]:
# Plot Poisson(lambda=3) pmf (truncated at k=14)
plt.stem(ks, probs_pois, use_line_collection=True)
plt.xlabel('k (count)')
plt.ylabel('P(X = k)')
plt.title('Poisson(λ=3) pmf (truncated)')
plt.show()

In [ ]:
# Simulate Poisson and compare mean/variance
lam = 3.0
samples_pois = np.random.poisson(lam=lam, size=50000)

mean_emp = samples_pois.mean()
var_emp = samples_pois.var(ddof=1)

print("Theoretical mean:", lam)
print("Empirical mean:", mean_emp)
print("Theoretical variance:", lam)
print("Empirical variance:", var_emp)

---
## 6. Comparing Discrete Distributions

In this section we'll put some of these distributions side-by-side so you can see how changing
parameters changes the shape.

Some classic relationships:
- Binomial$(n, p)$ with large $n$ and small $p$ can be approximated by Poisson$(\lambda = np)$.
- Geometric$(p)$ is related to repeated Bernoulli$(p)$ trials.

We'll do two quick visuals:
1. Binomial vs Poisson (with $\lambda = np$)
2. How Geometric changes as $p$ changes.

In [ ]:
# 6.1 Binomial vs Poisson approximation
n = 50
p = 0.1
lam = n * p

ks = np.arange(0, 30)
binom_probs = np.array([binomial_pmf(k, n, p) for k in ks])
pois_probs = np.array([poisson_pmf(k, lam) for k in ks])

plt.plot(ks, binom_probs, marker='o', linestyle='-', label='Binomial(n=50, p=0.1)')
plt.plot(ks, pois_probs, marker='x', linestyle='--', label='Poisson(λ=np=5)')
plt.xlabel('k')
plt.ylabel('P(X = k)')
plt.title('Binomial vs Poisson approximation')
plt.legend()
plt.show()

In [ ]:
# 6.2 Geometric for different p values
ks = np.arange(1, 15)
for p in [0.2, 0.5, 0.8]:
    probs = np.array([geometric_pmf(k, p) for k in ks])
    plt.plot(ks, probs, marker='o', label=f'p={p}')

plt.xlabel('k (trial of first success)')
plt.ylabel('P(X = k)')
plt.title('Geometric pmf for different p')
plt.legend()
plt.show()

---
## 7. Practice / TODOs

### 7.1 Suggested exercises (plug in your course questions)

- **Bernoulli / Binomial**
  - Work through tutorial questions about:
    - repeated coin tosses,
    - quality control (defective / non-defective),
    - number of successes out of $n$.

- **Geometric**
  - Model "number of attempts until first success" questions.
  - Example: number of customers until the first one who buys something.

- **Poisson**
  - Typical word problems: counts in a time window or spatial region.

### 7.2 Placeholders for worked examples

Below you can add cells with actual questions from your MATH4002 sheets.
Try to:
- first solve analytically (by hand or symbolic),
- then simulate the scenario to check your intuition.

---
**Example template (markdown cell):**

**Question:**  
Suppose emails arrive to an inbox according to a Poisson process with rate 4 per hour.  
What is the probability of receiving exactly 2 emails in the next hour?

**Solution (theory):**  
$X \sim \text{Poisson}(\lambda = 4)$.  
$P(X = 2) = e^{-4} 4^2 / 2!$.

**Example template (code cell):**

```python
# Check with our poisson_pmf function and by simulation
lam = 4
k = 2
p_theory = poisson_pmf(k, lam)

samples = np.random.poisson(lam=lam, size=100000)
p_emp = np.mean(samples == k)

p_theory, p_emp
```